# Aula 7 - Meio de transporte das mulheres por cor/raça (IBGE 2022)

Trabalho de análise de dados usando pandas, matplotlib e seaborn.

## Parte 1

In [ ]:
# 1 - importando as bibliotecas
# obs: se der erro de "modulo nao encontrado", descomenta a linha de baixo e roda
# !pip install matplotlib pandas seaborn openpyxl

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 2 - lendo o arquivo excel, aba "Exemplo grafico Brasil"
# a primeira linha do excel e so o titulo da tabela, entao pulo ela com skiprows=1
df = pd.read_excel('Tabela_13_Meio_de_transporte.xlsx', sheet_name='Exemplo gráfico Brasil', skiprows=1)
df

In [ ]:
# 3 - fazendo o piechart (grafico de pizza) para cada cor/raca

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

cores = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0', '#ffb3e6']

grupos = ['Branca', 'Preta ou parda', 'Indígena']

for i, col in enumerate(grupos):
    axes[i].pie(
        df[col],
        labels=df['Meio de transporte/cor ou raça'],
        autopct='%1.3f%%',  # 3 casas decimais, como pedido no exercicio
        startangle=180,     # angulo inicial de rotacao
        colors=cores,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.2},
    )
    axes[i].set_title(f'Cor/Raça: {col}', fontsize=12, fontweight='bold')

plt.suptitle(
    'Meio de transporte usado por mulheres para ir ao trabalho, por cor/raça - Brasil 2022',
    fontsize=14,
    fontweight='bold',
)
plt.tight_layout()
plt.savefig('grafico_transporte.png', dpi=300)
plt.show()

In [ ]:
# calculando a diferenca entre as mulheres brancas e as mulheres pretas ou pardas
# em cada meio de transporte (so pra ajudar a escrever a analise depois)

df_calc = df.set_index('Meio de transporte/cor ou raça')

for modo in df_calc.index:
    branca = df_calc.loc[modo, 'Branca']
    preta = df_calc.loc[modo, 'Preta ou parda']
    diferenca = branca - preta
    print(f'{modo}: {branca}% - {preta}% = {diferenca:.1f} pontos percentuais')

### Análise dos gráficos

Comparando as mulheres brancas com as mulheres pretas ou pardas (Branca - Preta ou parda) em cada meio de transporte:

- **A pé:** 19.4% - 24.8% = **-5.4%** (pretas ou pardas andam mais a pé)
- **Bicicleta:** 3.2% - 5.2% = **-2.0%** (pretas ou pardas usam mais bicicleta)
- **Motocicleta/Mototáxi:** 9.9% - 14.2% = **-4.3%** (pretas ou pardas usam mais moto)
- **Automóvel, táxi ou assemelhados:** 41.8% - 20.6% = **+21.2%** (brancas usam muito mais carro)
- **Transporte coletivo:** 25.2% - 34.6% = **-9.4%** (pretas ou pardas usam mais ônibus/metrô)
- **Outros:** 0.5% - 0.6% = **-0.1%** (bem parecido)

Dá pra ver que o maior contraste é no uso de automóvel: as mulheres brancas usam carro bem mais que as mulheres pretas ou pardas (diferença de mais de 21 pontos percentuais). Já nos outros meios de transporte (principalmente a pé, transporte coletivo e moto) quem mais usa são as mulheres pretas ou pardas. As mulheres indígenas se destacam bastante no "a pé" (37.5%), bem acima das outras duas cores/raças.

Isso mostra que o acesso a carro próprio provavelmente está ligado a uma questão de renda, que no Brasil ainda é bem desigual entre as raças.

## Parte 2

In [ ]:
# 1 - lendo o arquivo excel, aba "BR GR UF MU"
# header=None porque o cabecalho dessa aba tem 3 linhas (fica mais facil montar na mao)
df2 = pd.read_excel('Tabela_13_Meio_de_transporte.xlsx', sheet_name='BR GR UF MU', header=None)
df2.head(10)

In [ ]:
# montando os nomes das colunas na mao, seguindo o padrao do cabecalho:
# Local | Branca (6 modos) | Preta ou Parda (6 modos) | Indigena (6 modos)

colunas = ['Local']
racas = ['Branca', 'Preta_Parda', 'Indigena']
modos = ['A_pe', 'Bicicleta', 'Moto', 'Automovel', 'Transporte_coletivo', 'Outros']

for raca in racas:
    for modo in modos:
        colunas.append(f'{raca}_{modo}')

df2.columns = colunas

# as 3 primeiras linhas eram o cabecalho, entao tiro elas
df2 = df2.iloc[3:].reset_index(drop=True)

# transformando as colunas de numero em numero mesmo (as vezes vem como texto)
for c in colunas[1:]:
    df2[c] = pd.to_numeric(df2[c], errors='coerce')

df2.head()

**Como a tabela é organizada:** a primeira linha é o Brasil, depois vêm as 5 grandes regiões, depois os 27 estados (26 estados + Distrito Federal) e depois todos os municípios do Brasil (com a sigla do estado entre parênteses, tipo "Cacoal (RO)"). Por isso consigo separar cada pedaço usando `.iloc[]`.

In [ ]:
# 2 - listando apenas os estados
# linha 0 = Brasil, linhas 1 a 5 = regioes, linhas 6 a 32 = os 27 estados
estados = df2.iloc[6:33].reset_index(drop=True)

print(f'Total de estados: {len(estados)}')
print(estados['Local'].tolist())

In [ ]:
# separando tambem os municipios (o resto da tabela, depois dos estados)
municipios = df2.iloc[33:].reset_index(drop=True)
print(f'Total de municipios: {len(municipios)}')
municipios.head()

### 3 - Respondendo as perguntas

Para responder "em geral" eu fiz a média simples entre as 3 colunas de cor/raça (Branca, Preta ou Parda e Indígena) de cada meio de transporte. Não é um cálculo perfeito (o ideal seria uma média ponderada pela população), mas serve bem pra essa atividade.

In [ ]:
# Pergunta 1 e 2: estado com MAIS e com MENOS mulheres usando transporte coletivo (em geral)

estados['coletivo_geral'] = estados[[
    'Branca_Transporte_coletivo',
    'Preta_Parda_Transporte_coletivo',
    'Indigena_Transporte_coletivo'
]].mean(axis=1)

mais_coletivo = estados.loc[estados['coletivo_geral'].idxmax()]
menos_coletivo = estados.loc[estados['coletivo_geral'].idxmin()]

print(f"Estado com MAIS mulheres usando transporte coletivo: {mais_coletivo['Local']} ({mais_coletivo['coletivo_geral']:.1f}%)")
print(f"Estado com MENOS mulheres usando transporte coletivo: {menos_coletivo['Local']} ({menos_coletivo['coletivo_geral']:.1f}%)")

In [ ]:
# Pergunta 3: cidade do Brasil com mais mulheres andando de bicicleta

municipios['bici_geral'] = municipios[[
    'Branca_Bicicleta',
    'Preta_Parda_Bicicleta',
    'Indigena_Bicicleta'
]].mean(axis=1)

cidade_bici = municipios.loc[municipios['bici_geral'].idxmax()]
print(f"Cidade com mais mulheres andando de bicicleta: {cidade_bici['Local']} ({cidade_bici['bici_geral']:.1f}%)")

In [ ]:
# Pergunta 4: cidade do Brasil com mais mulheres utilizando carro

municipios['carro_geral'] = municipios[[
    'Branca_Automovel',
    'Preta_Parda_Automovel',
    'Indigena_Automovel'
]].mean(axis=1)

cidade_carro = municipios.loc[municipios['carro_geral'].idxmax()]
print(f"Cidade com mais mulheres utilizando carro: {cidade_carro['Local']} ({cidade_carro['carro_geral']:.1f}%)")

### Respostas finais

- **Estado com mais mulheres no transporte coletivo:** Rio de Janeiro (≈ 52.2%)
- **Estado com menos mulheres no transporte coletivo:** Rondônia (≈ 4.3%)
- **Cidade com mais mulheres andando de bicicleta:** Tuiuti (SP) (100%)
- **Cidade com mais mulheres usando carro:** Bom Jesus do Oeste (SC) (≈ 86.1%)

Obs: como algumas cidades pequenas têm poucas pessoas entrevistadas na pesquisa, é normal aparecer valor de 100% em uma cidade bem pequena (tipo Tuiuti) — não significa que é uma tendência forte, só que na amostra daquela cidade praticamente todas as mulheres responderam isso.